## Installs

In [2]:
RUNINSTALLS = False

if RUNINSTALLS:
  !pip install openpyxl
  !pip install --upgrade google-cloud-aiplatform


## Notebook Setup

In [3]:
from IPython.display import HTML, display

def set_css(arg=None):
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

import logging
import sys
format_string = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
logger = logging.getLogger()
fhandler = logging.FileHandler(filename='notebook.log', mode='a')
formatter = logging.Formatter(format_string)
fhandler.setFormatter(formatter)
#logger.addHandler(fhandler)  #uncomment if you want a log file
logging.basicConfig(format=format_string,
                     level=logging.INFO, stream=sys.stdout)
logger.setLevel(logging.INFO)


## Imports

In [4]:
from google.cloud import storage
import pandas as pd
import openpyxl
import base64
import vertexai
import yaml, json
from vertexai.generative_models import GenerationConfig, GenerativeModel, Part, FinishReason
import vertexai.preview.generative_models as generative_models
from io import BytesIO

## Project setup

In [5]:
BUCKET_NAME = "uk-bh-experiments-argolis-us"
FOLDER_PATH = "subsea7/hseq_data/"

In [25]:

def get_description_column(df):
  column_names = list(df)
  for column_name in column_names:
      if "desc" in column_name.lower():
         return column_name
  #if none found, return the 1st column
  return column_names[0]


def get_excel_sheet( file_name, sheet_name=None, header=0, mandatory_columns=None):

  storage_client = storage.Client()
  bucket = storage_client.bucket(BUCKET_NAME)
  blob = bucket.blob(file_name)
  logging.info(f"Got file {file_name}")

  with blob.open("rb") as f:
      file_bytes = BytesIO(f.read())

  logging.info(f"read file {file_name}")

  openpyxl.reader.excel.warnings.simplefilter(action='ignore')
  gs_uri = f"gs://{BUCKET_NAME}/{FOLDER_PATH}/{file_name}"

  if sheet_name is None:
    #xls = pd.ExcelFile(gs_uri,sheetname=None, nrows=0, engine='openpyxl')
    sheet_names = pd.ExcelFile(file_bytes,  engine='openpyxl').sheet_names
    print(f"Available sheets:")
    for sheet_name  in sheet_names:
        print(f"{sheet_name}")
    return sheet_names
  else:
    print(f"Reading sheet {sheet_name}")

    with pd.ExcelFile(file_bytes) as xls:
      df = pd.read_excel(xls, sheet_name, header=header)
      print(f"Sheet {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")
      if mandatory_columns == None:
         mandatory_columns = [get_description_column(df)]
      logging.info(f"Dropping all rows that have nothing in the columns: {mandatory_columns}")
      df.dropna(subset=mandatory_columns, inplace=True)

      print(f"Cleaned Sheet {sheet_name} has {df.shape[0]} rows and {df.shape[1]} columns")

    return df


def df_to_json(df):
  sheet_json_str = df.to_json(orient='records')
  #print(f'{sheet_json_str[:1]}')
  sheet_json = json.loads(sheet_json_str)
  return sheet_json_str

## Use JSON as TXT

In [36]:
def generate_using_json(documents, prompt):
    vertexai.init(project="uk-bh-experiments-argolis", location="us-central1")
    model = GenerativeModel("gemini-1.5-flash-001")
    generation_config=GenerationConfig()

    parts = [Part.from_text(document) for document in documents]
    parts.append(prompt)
    response = model.count_tokens(parts)
    print(f"Prompt Token Count: {response.total_tokens}")
    print(f"Prompt Character Count: {response.total_billable_characters}")

    response = model.generate_content(parts,generation_config=generation_config,stream=False)

    # Response tokens count
    usage_metadata = response.usage_metadata
    print(f"Prompt Token Count: {usage_metadata.prompt_token_count}")
    print(f"Candidates Token Count: {usage_metadata.candidates_token_count}")
    print(f"Total Token Count: {usage_metadata.total_token_count}")



    response_text = response.text

    return response_text


In [ ]:
df1_sheets = get_excel_sheet("subsea7/hseq_data/SevenArctic.xlsx")
print(f"{df1_sheets}")

In [30]:
df2 = get_excel_sheet("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1)

2024-07-12 10:12:47,381 - root - INFO - Got file subsea7/hseq_data/NormandSubsea.xlsx
2024-07-12 10:12:48,042 - root - INFO - read file subsea7/hseq_data/NormandSubsea.xlsx
Reading sheet OBSERVATIONS
Sheet OBSERVATIONS has 219 rows and 61 columns
2024-07-12 10:12:51,726 - root - INFO - Dropping all rows that have nothing in the columns: ['Description of observation']
Cleaned Sheet OBSERVATIONS has 34 rows and 61 columns


In [40]:


#documents = df_to_json(df2)
sheet_json_str = df2.to_json(orient='records')
sheet_json = json.loads(sheet_json_str)
document1 = Part.from_text(json.dumps(sheet_json))

observation = generate_using_json([sheet_json_str], "what is the most unsafe location")
print(f'{observation}')

I0000 00:00:1720776140.702354   28855 tcp_posix.cc:809] IOMGR endpoint shutdown
I0000 00:00:1720776165.702309   28855 tcp_posix.cc:809] IOMGR endpoint shutdown


Prompt Token Count: 16765
Prompt Character Count: 64132


TypeError: Unexpected item type: [text: "[{\"Obs. No\":120.0,\"Date\":1609545600000,\"Name of observer\":\"Brian Bullock\",\"Department\":\"Project - Deck\",\"Description of observation\":\"Suggestion to weld a padeye directly below MHS main lift.  This will eliminate trip hazards and give a direct seafastening point.\",\"Immediate Action\":\"Spoke with the welder to look into modifying grating to suit padeye.\",\"Further Action Required\":\"This will be reviewed first by the Deck Fmn and OM before any actions are taken. \",\"Status\":\"Closed\",\"Observation close out comment\\/date input\":1614816000000,\"Unnamed: 9\":null,\"Eyes on path\":null,\"Line of Fire\":null,\"Use of tools & Equip\":null,\"Pinch Points\":null,\"3-Point contact\":null,\"Communication\":null,\"Housekeeping\":null,\"Pre Job Planning\":null,\"Assistance need\\/used\":null,\"Walking\\/working surfaces\":null,\"Eyes on task\":null,\"Hot Work preparation\":null,\"Manual handling\":null,\"Isolation systems\":null,\"u

In [ ]:
df2 = get_excel_sheet("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1)
df3 = get_excel_sheet(f"{FOLDER_PATH}SeawayStrashnov.xlsx", "Obs_Int Register", 1)

In [43]:
df1 = get_excel_sheet("subsea7/hseq_data/NormandSubsea.xlsx", "OBSERVATIONS", 1)
df_json = df_to_json(df1)


observation = generate_using_json([df_json], "what is the most unsafe location")

Reading sheet OBSERVATIONS
Sheet OBSERVATIONS has 219 rows and 61 columns
Cleaned Sheet OBSERVATIONS has 34 rows and 61 columns


AttributeError: 'list' object has no attribute 'loads'

In [ ]:
print(observation)

## Try as YAML

In [ ]:

df_yaml = yaml.dump(df1.to_dict(orient='records'),default_flow_style=None)

In [ ]:
print(f'{df_yaml[:1000]}')

In [ ]:

document1 = Part.from_text(df_yaml)


observation = generate(document1, "what is the most unsafe location")

In [ ]:
print(observation)